In [1]:
import warnings
# Ignore all FutureWarnings
warnings.filterwarnings("ignore", category=FutureWarning)

import re
import math
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn import tree
from sklearn.model_selection import RandomizedSearchCV, train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from sklearn.svm import SVC
import random
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import spacy
from spacy.lang.en.stop_words import STOP_WORDS

# Make results reproducible
random.seed(100)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\UFC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
# Labelled data loading
data = pd.read_csv('A2_customer_churn_labeled.csv')
data.head()

,ID,credit_score,tenure,balance,number_of_products,has_credit_card,is_active_member,salary,customer_profile,Y
0,0,585,4,0.00,2,0,1,101728.46,This customer is a 44-year-old female from Spa...,0
1,1,743,6,140348.56,2,1,1,163254.39,This customer is a 32-year-old female from Ger...,0
2,2,527,10,136733.23,1,1,1,57589.29,This customer is a 41-year-old female from Ger...,0
3,3,732,6,98792.40,1,1,0,81491.70,This customer is a 45-year-old female from Ger...,1
4,4,641,3,0.00,2,1,0,116466.19,This customer is a 38-year-old female from Fra...,0


In [3]:
print('Shape of labeled data: ', data.shape)

Shape of labeled data:  (7000, 10)


In [4]:
def get_last_two_sentences(text):
    sentences = sent_tokenize(text)

    # Get the last 2 sentences
    last_two_sentences = sentences[-2:]

    return ' '.join(last_two_sentences)

# Create an object instance sih of SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

# Function that returns compound polarity score of the text
def get_polarity(text):
    # Get the polarity scores of the passed text
    return sia.polarity_scores(text)['compound']

# Load spaCy's English tokenizer and tagger
nlp = spacy.load("en_core_web_sm")

# Define a function to perform tokenization, stopwords removal, and lemmatization
def preprocess_text(text):
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if token.text.lower() not in STOP_WORDS]
    return " ".join(tokens)


In [5]:
def add_new_columns(dataset):
     #  Just comment the line for the column that is not needed
    
    dataset['age'] = dataset['customer_profile'].apply(lambda x: int(re.findall('(\d+)-year-old', x)[0]))
    dataset['gender'] = dataset['customer_profile'].apply(lambda x: re.findall('(male|female)', x)[0])
    dataset['country'] = dataset['customer_profile'].apply(lambda x: re.findall('from (\w+)', x)[0])
    dataset['customer_profile_tokenized'] = dataset['customer_profile'].apply(lambda text: preprocess_text(text))
    return dataset

In [6]:
data1 = add_new_columns(data)

In [7]:
def drop_any_existing_columns(dataset, columns = ['ID']):
    dataset = dataset.drop(columns=columns, inplace=False)
    return dataset

In [8]:
columns_to_drop = ['ID']

data2 = drop_any_existing_columns(data1, columns = columns_to_drop)

In [9]:
data2

,credit_score,tenure,balance,number_of_products,has_credit_card,is_active_member,salary,customer_profile,Y,age,gender,country,customer_profile_tokenized
0,585,4,0.00,2,0,1,101728.46,This customer is a 44-year-old female from Spa...,0,44,female,Spain,customer 44 - year - old female Spain credit s...
1,743,6,140348.56,2,1,1,163254.39,This customer is a 32-year-old female from Ger...,0,32,female,Germany,customer 32 - year - old female Germany credit...
2,527,10,136733.23,1,1,1,57589.29,This customer is a 41-year-old female from Ger...,0,41,female,Germany,customer 41 - year - old female Germany credit...
3,732,6,98792.40,1,1,0,81491.70,This customer is a 45-year-old female from Ger...,1,45,female,Germany,customer 45 - year - old female Germany credit...
4,641,3,0.00,2,1,0,116466.19,This customer is a 38-year-old female from Fra...,0,38,female,France,customer 38 - year - old female France credit ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6995,778,1,151958.19,3,1,1,131238.38,This customer is a 35-year-old female from Ger...,1,35,female,Germany,customer 35 - year - old female Germany credit...
6996,781,5,0.00,2,0,0,72969.90,This customer is a 27-year-old female from Fra...,0,27,female,France,customer 27 - year - old female France credit ...
6997,738,2,0.00,2,1,1,170421.12,This customer is a 29-year-old male from Franc...,0,29,male,France,customer 29 - year - old male France credit sc...
6998,512,5,0.00,2,1,1,146457.83,This customer is a 40-year-old male from Franc...,0,40,male,France,customer 40 - year - old male France credit sc...


In [59]:
def extract_balance(txt):
    if re.search(r'balance\s*(\d+,?\d*[.]\d+)', txt) is not None:
        balance = re.findall(r'balance\s*(\d+,?\d*[.]\d+)', txt)[0]
        return round(float(re.sub(',', '', balance)), 2) 
    if re.search(r'balance\s*currently\s*(\d+,?\d*[.]\d+)', txt) is not None:
        balance = re.findall(r'balance\s*currently\s*(\d+,?\d*[.]\d+)', txt)[0]
        return round(float(re.sub(',', '', balance)), 2)
    else: 
        return 0.0

In [84]:
data2['balance_profile'] = data2['customer_profile_tokenized'].apply(lambda txt: extract_balance(txt))

In [85]:
unmatched = data2[data2['balance'] != data2['balance_profile']]
unmatched

,credit_score,tenure,balance,number_of_products,has_credit_card,is_active_member,salary,customer_profile,Y,age,gender,country,customer_profile_tokenized,balance_profile
106,775,6,179886.4,2,0,0,153122.58,This customer is a 38-year-old female from Ger...,0,38,female,Germany,customer 38 - year - old female Germany credit...,179886.41
140,688,7,138162.4,2,1,1,113926.31,This customer is a 37-year-old male from Spain...,0,37,male,Spain,customer 37 - year - old male Spain credit sco...,138162.41
238,687,9,135962.4,2,1,0,121747.96,This customer is a 33-year-old male from Germa...,0,33,male,Germany,customer 33 - year - old male Germany credit s...,135962.41
248,686,9,141918.1,2,0,1,184036.47,This customer is a 33-year-old male from Germa...,0,33,male,Germany,customer 33 - year - old male Germany credit s...,141918.09
364,684,4,139723.9,1,1,1,120612.11,This customer is a 39-year-old female from Spa...,0,39,female,Spain,customer 39 - year - old female Spain credit s...,139723.91
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6809,542,8,139330.1,1,0,0,54372.37,This customer is a 25-year-old male from Germa...,0,25,male,Germany,customer 25 - year - old male Germany credit s...,139330.09
6810,597,6,135703.6,2,0,0,74850.84,This customer is a 33-year-old female from Fra...,0,33,female,France,customer 33 - year - old female France credit ...,135703.59
6815,528,3,156687.1,1,1,0,199320.77,This customer is a 35-year-old male from Franc...,0,35,male,France,customer 35 - year - old male France credit sc...,156687.09
6825,524,2,180516.9,1,1,0,180002.42,This customer is a 40-year-old male from Franc...,0,40,male,France,customer 40 - year - old male France credit sc...,180516.91


In [86]:
data2.loc[unmatched.index, 'balance_profile'] = data2.loc[unmatched.index, 'balance_profile'].apply(lambda b: round(b,1))

In [87]:
unmatched = data2[data2['balance'] != data2['balance_profile']]
unmatched

,credit_score,tenure,balance,number_of_products,has_credit_card,is_active_member,salary,customer_profile,Y,age,gender,country,customer_profile_tokenized,balance_profile
1217,584,1,0.0,1,0,1,152567.75,This customer is a 50-year-old female from Spa...,1,50,female,Spain,customer 50 - year - old female Spain credit s...,1.0


In [48]:
for txt in data2[data2['balance_profile'].isnull()][['customer_profile']].values:
    print(txt)

In [15]:
re.findall(r'balance\s*(\d+.\d+)', data.loc[0, 'customer_profile_tokenized'])

['0.0']

In [92]:
data2['seq_len'] =  data['customer_profile'].apply(lambda x: len(x.split(' ')))

In [93]:
max(data2['seq_len'])

70